In [1]:
import seaborn as sns
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [2]:
df=sns.load_dataset('iris')
df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


### Encode the Target Variable

In [3]:
le=LabelEncoder()
df['species']=le.fit_transform(df['species'])

### Seperate Dependent and Independent Variable

In [4]:
x=df.drop('species',axis=1)
y=df['species']

### Split the train and test data

In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV


In [6]:
X_train,X_test,Y_train,Y_test= train_test_split(x,y,test_size=0.33)

### Use Pipeline

In [7]:
from sklearn.pipeline import Pipeline

In [8]:
pipeline=Pipeline([
    ('classifier',LogisticRegression())
])

In [9]:
Search_space=[
    {
        'classifier':[LogisticRegression(max_iter=500)],
        'classifier__C':[0.1,1,10],
        'classifier__solver':['liblinear','lbfgs']
    },
    {
        'classifier':[SVC()],
        'classifier__C':[0.1,1,10],
        'classifier__kernel':['linear','rbf']
    },
    {
        'classifier': [DecisionTreeClassifier()],
        'classifier__max_depth': [3, 5, None],
        'classifier__criterion': ['gini', 'entropy']
    },
    {
        'classifier': [RandomForestClassifier()],
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [3, 5, None]
    }
]

In [10]:
grid=GridSearchCV(pipeline,Search_space,cv=5,scoring='accuracy')

In [11]:
print ("Mode and Parameter Selection")
grid.fit(X_train,Y_train)

Mode and Parameter Selection


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('classifier', LogisticRegression())]),
             param_grid=[{'classifier': [LogisticRegression(max_iter=500)],
                          'classifier__C': [0.1, 1, 10],
                          'classifier__solver': ['liblinear', 'lbfgs']},
                         {'classifier': [SVC()], 'classifier__C': [0.1, 1, 10],
                          'classifier__kernel': ['linear', 'rbf']},
                         {'classifier': [DecisionTreeClassifier()],
                          'classifier__criterion': ['gini', 'entropy'],
                          'classifier__max_depth': [3, 5, None]},
                         {'classifier': [RandomForestClassifier()],
                          'classifier__max_depth': [3, 5, None],
                          'classifier__n_estimators': [50, 100, 200]}],
             scoring='accuracy')

In [12]:
print(f"Best Algo:{grid.best_params_['classifier']}")
print(f"Best Parameter:{grid.best_params_}")
print(f"Best Accuracy:{grid.best_score_:.4f}")

Best Algo:SVC()
Best Parameter:{'classifier': SVC(), 'classifier__C': 1, 'classifier__kernel': 'linear'}
Best Accuracy:0.9700


In [13]:
from sklearn.metrics import accuracy_score,classification_report

In [14]:
y_pred=grid.predict(X_test)
final_acc=accuracy_score(Y_test,y_pred)

In [15]:
print(f"Test Accuracy: {final_acc:.4f}")
print("\nClassification Report:")
print(classification_report(Y_test, y_pred, target_names=le.classes_))

Test Accuracy: 0.9800

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        16
  versicolor       0.94      1.00      0.97        17
   virginica       1.00      0.94      0.97        17

    accuracy                           0.98        50
   macro avg       0.98      0.98      0.98        50
weighted avg       0.98      0.98      0.98        50



In [16]:
import pickle

# Save the model
with open('iris_model.pkl', 'wb') as f:
    pickle.dump(grid.best_estimator_, f)

print("Model saved successfully!")

Model saved successfully!
